# Attention：从透明 oracle 到框架对账

**适合读者**：已经理解矩阵乘法、准备把 Attention 公式和框架实现对上的学习者。

**先修**：softmax、张量 shape；需要 NumPy、PyTorch 与 JAX，不需要 GPU。运行前请按 [Notebook 环境说明](README.md) 安装 profile，并运行 `python scripts/doctor.py --profile notebooks`。

**路线**：先预测 causal mask → 查看 NumPy oracle → PyTorch/JAX 对账 → 移除 mask/缩放，并制造 fully masked row 反例。

**完成信号**：能解释右上三角为什么必须为零、fully masked row 为什么必须显式处理，以及三框架数值一致为什么不等于 kernel 性能一致。

## 1. 先预测，再看 NumPy oracle

Attention 在这里按 $S=QK^T/\sqrt{d_k}$、$P=\operatorname{softmax}(S+M)$、$O=PV$ 计算：score 矩阵的行是 query、列是 key；`mask` 为 `True` 表示该 key 可见，`False` 的位置在 softmax 前变成 $-\infty$。

先手算一个两 token 的例子。第二个 query 看见两个 key，它的两个 scaled scores 是 $0$ 与 $\ln 3$，所以概率应是 $1/4, 3/4$，value 为 $2,6$ 时输出应为 $5$。接着画一个 4×4 矩阵：第 0 行应看见几个 key？第 3 行呢？再预测右上三角的概率应该是 0、很小，还是不确定。

这里的 oracle 是可逐步检查、能用小输入手算的参考计算；它为后面的 PyTorch/JAX 重写提供判定依据，而不代表 GPU kernel 的内存或速度。固定随机输入则用来把这个判定延伸到一个完整 probability matrix。

In [ ]:
import numpy as np

from about_llm.from_scratch.attention_numpy import causal_mask, scaled_dot_product_attention

toy_query = np.array([[[1.0, 0.0], [1.0, 0.0]]])
toy_key = np.array([[[0.0, 0.0], [np.sqrt(2.0) * np.log(3.0), 0.0]]])
toy_value = np.array([[[2.0], [6.0]]])
toy_output, toy_probabilities = scaled_dot_product_attention(
    toy_query, toy_key, toy_value, mask=causal_mask(2)
)
np.testing.assert_allclose(toy_probabilities[0, 1], [0.25, 0.75])
np.testing.assert_allclose(toy_output[0, 1], [5.0])
print('hand calculation:', toy_probabilities[0, 1], toy_output[0, 1])

rng = np.random.default_rng(42)
head_dim = 8
query = rng.normal(size=(1, 4, head_dim)).astype(np.float32)
key = rng.normal(size=(1, 4, head_dim)).astype(np.float32)
value = rng.normal(size=(1, 4, 6)).astype(np.float32)
mask = causal_mask(4)
raw_scores = query @ np.swapaxes(key, -2, -1)
scaled_scores = raw_scores / (head_dim ** 0.5)
numpy_output, numpy_probabilities = scaled_dot_product_attention(query, key, value, mask=mask)
print('mask:\n', mask.astype(int))
print('probabilities:\n', np.round(numpy_probabilities[0], 3))
assert np.all(numpy_probabilities[0][np.triu_indices(4, k=1)] == 0)


## 2. 用 PyTorch 重写同一公式

逐项对照 score shape、缩放、mask 方向和 softmax 轴。若结果不同，先定位第一处张量差异，不要立刻归因于“框架精度”。

In [ ]:
import torch

q_t, k_t, v_t = map(torch.from_numpy, (query, key, value))
raw_scores_t = q_t @ k_t.transpose(-2, -1)
scaled_scores_t = raw_scores_t / (q_t.shape[-1] ** 0.5)
scores_t = scaled_scores_t.masked_fill(~torch.from_numpy(mask), -torch.inf)
torch_output = torch.softmax(scores_t, dim=-1) @ v_t
np.testing.assert_allclose(torch_output.numpy(), numpy_output, rtol=1e-5, atol=1e-6)
print('PyTorch matches NumPy:', torch_output.shape)


## 3. 再换 JAX，但不改变问题

JAX 使用同一组 Q/K/V 和同一个 NumPy mask。backend 是实际执行数组计算的实现，dtype 是数组的数值格式，fused kernel 是把多个算子合并执行的实现。三方一致只证明这个固定输入下的前向公式对齐，不证明这些 backend、dtype 或 fused kernel 等价。

In [ ]:
import jax
import jax.numpy as jnp

raw_scores_j = jnp.asarray(query) @ jnp.swapaxes(jnp.asarray(key), -2, -1)
scaled_scores_j = raw_scores_j / (query.shape[-1] ** 0.5)
scores_j = jnp.where(jnp.asarray(mask), scaled_scores_j, -jnp.inf)
jax_output = jax.nn.softmax(scores_j, axis=-1) @ jnp.asarray(value)
np.testing.assert_allclose(np.asarray(jax_output), numpy_output, rtol=1e-5, atol=1e-6)
print('JAX matches NumPy:', jax_output.shape)


## 4. 故意破坏两个关键条件

先移除 causal mask，观察未来概率质量；再保留 mask 但移除 1/√d 缩放，观察概率分布变化。先写下你的预测，再运行。

In [ ]:
_, unmasked_probabilities = scaled_dot_product_attention(query, key, value)
future_mass = float(unmasked_probabilities[0][np.triu_indices(4, k=1)].sum())

unscaled_scores = raw_scores.copy()
masked_unscaled_scores = np.where(mask, unscaled_scores, -np.inf)
shifted_unscaled_scores = masked_unscaled_scores - np.max(
    masked_unscaled_scores, axis=-1, keepdims=True
)
unscaled_probabilities = np.exp(shifted_unscaled_scores)
unscaled_probabilities /= unscaled_probabilities.sum(axis=-1, keepdims=True)
scale_delta = float(np.max(np.abs(unscaled_probabilities - numpy_probabilities)))

def entropy_without_zero_probabilities(probabilities):
    nonzero = probabilities[probabilities > 0]
    return float(-(nonzero * np.log(nonzero)).sum())

comparison_query = 2
scaled_entropy = entropy_without_zero_probabilities(
    numpy_probabilities[0, comparison_query]
)
unscaled_entropy = entropy_without_zero_probabilities(
    unscaled_probabilities[0, comparison_query]
)

fully_masked = mask.copy()
fully_masked[-1] = False
try:
    scaled_dot_product_attention(query, key, value, mask=fully_masked)
except ValueError as error:
    print('NumPy oracle rejects a fully masked row:', error)
else:
    raise AssertionError('the NumPy oracle must reject a fully masked row')

torch_fully_masked_scores = q_t @ k_t.transpose(-2, -1) / (q_t.shape[-1] ** 0.5)
torch_fully_masked_scores = torch_fully_masked_scores.masked_fill(
    ~torch.from_numpy(fully_masked), -torch.inf
)
torch_fully_masked_probabilities = torch.softmax(torch_fully_masked_scores, dim=-1)
assert not torch.isfinite(torch_fully_masked_probabilities).all()

jax_fully_masked_scores = jnp.where(
    jnp.asarray(fully_masked), scores_j, -jnp.inf
)
jax_fully_masked_probabilities = jax.nn.softmax(jax_fully_masked_scores, axis=-1)
assert not np.asarray(jnp.isfinite(jax_fully_masked_probabilities)).all()

print('future probability mass without causal mask:', round(future_mass, 6))
print('max probability change without 1/sqrt(d):', round(scale_delta, 6))
print('raw/scaled score variance:', round(raw_scores.var(), 6), round(scaled_scores.var(), 6))
print('scaled/unscaled entropy for query row', comparison_query, ':',
      round(scaled_entropy, 6), round(unscaled_entropy, 6))
# 成对对照是：同一 Q/K/V 有 mask 时未来位置严格为 0；移除同一个 mask 后，
# 此固定种子、shape 和 NumPy 实现的 fixture 中 future_mass 应落在 (0.1, 2.0)。
# 该范围只防止这个反例退化，不是任意输入下 future_mass 的理论阈值。
masked_future_mass = float(numpy_probabilities[0][np.triu_indices(4, k=1)].sum())
assert masked_future_mass == 0.0
assert 0.1 < future_mass < 2.0
assert scale_delta > 1e-3

## 5. 解释结果，而不只记录“通过”

- mask 反例说明 causal 不变量必须靠测试锁定；shape 正确并不足够。
- 缩放反例说明 1/√d 会改变 softmax 尺度，但单个随机样例不能证明训练稳定性。
- 三框架对账是数值正确性证据，不是 GPU 性能证据。

**练习**：改变 `head_dim`，先预测缩放前后 score 方差和 softmax entropy，再运行；随后预测 fully masked row 的三条路径：NumPy oracle 会拒绝，PyTorch/JAX 的 softmax 会出现非有限值。

**答案脚手架**：`isfinite` 检查结果是否不是 NaN 或正负无穷。分别记录有限的 `raw_scores.var()`、`scaled_scores.var()`，以及同一 `comparison_query` 行的 `scaled_entropy` 和 `unscaled_entropy`；不要对 `scores_t` 或 `masked_unscaled_scores` 求方差，它们为了 mask 已含 `-inf`。也不要直接计算 `-(p * log(p)).sum()`，因为 causal mask 的零概率会形成 `0 * log(0)`。head dimension 增大时，不缩放 score 的方差通常变大；fully masked row 必须有显式策略，不能把它当作普通有效行。